# LangGraph and LangSmith - Agentic RAG Powered by LangChain

In the following notebook we'll complete the following tasks:

- 🤝 Breakout Room #1:
  1. Install required libraries
  2. Set Environment Variables
  3. Creating our Tool Belt
  4. Creating Our State
  5. Creating and Compiling A Graph!

- 🤝 Breakout Room #2:
  1. Evaluating the LangGraph Application with LangSmith
  2. Adding Helpfulness Check and "Loop" Limits
  3. LangGraph for the "Patterns" of GenAI

# 🤝 Breakout Room #1

## Part 1: LangGraph - Building Cyclic Applications with LangChain

LangGraph is a tool that leverages LangChain Expression Language to build coordinated multi-actor and stateful applications that includes cyclic behaviour.

### Why Cycles?

In essence, we can think of a cycle in our graph as a more robust and customizable loop. It allows us to keep our application agent-forward while still giving the powerful functionality of traditional loops.

Due to the inclusion of cycles over loops, we can also compose rather complex flows through our graph in a much more readable and natural fashion. Effectively allowing us to recreate application flowcharts in code in an almost 1-to-1 fashion.

### Why LangGraph?

Beyond the agent-forward approach - we can easily compose and combine traditional "DAG" (directed acyclic graph) chains with powerful cyclic behaviour due to the tight integration with LCEL. This means it's a natural extension to LangChain's core offerings!

## Task 1:  Dependencies


## Task 2: Environment Variables

We'll want to set both our OpenAI API key and our LangSmith environment variables.

In [40]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [41]:
os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY")

In [42]:
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIE7 - LangGraph - {uuid4().hex[0:8]}"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")

## Task 3: Creating our Tool Belt

As is usually the case, we'll want to equip our agent with a toolbelt to help answer questions and add external knowledge.

There's a tonne of tools in the [LangChain Community Repo](https://github.com/langchain-ai/langchain-community/tree/main/libs/community) but we'll stick to a couple just so we can observe the cyclic nature of LangGraph in action!

We'll leverage:

- [Tavily Search Results](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/tavily_search/tool.py)
- [Arxiv](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/arxiv/tool.py)

#### 🏗️ Activity #1:

Please add the tools to use into our toolbelt.

> NOTE: Each tool in our toolbelt should be a method.

##### ✅ Answer:
I think the code below is already adding the tools Arxiv and Tavily into our toolbelt.<br>
Each element in the toolbelt is an object, because it is instantiating the class, and then we will be able to use the methods each class has.

In [4]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.arxiv.tool import ArxivQueryRun

tavily_tool = TavilySearchResults(max_results=5)

tool_belt = [
    tavily_tool,
    ArxivQueryRun(),
]

/tmp/ipykernel_6508/1203815797.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5)


### Model

Now we can set-up our model! We'll leverage the familiar OpenAI model suite for this example - but it's not *necessary* to use with LangGraph. LangGraph supports all models - though you might not find success with smaller models - as such, they recommend you stick with:

- OpenAI's GPT-3.5 and GPT-4
- Anthropic's Claude
- Google's Gemini

> NOTE: Because we're leveraging the OpenAI function calling API - we'll need to use OpenAI *for this specific example* (or any other service that exposes an OpenAI-style function calling API.

In [43]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

Now that we have our model set-up, let's "put on the tool belt", which is to say: We'll bind our LangChain formatted tools to the model in an OpenAI function calling format.

In [44]:
model = model.bind_tools(tool_belt)

#### ❓ Question #1:

How does the model determine which tool to use?

##### ✅ Answer:

With the command above `model = model.bind_tools(tool_belt)` we are telling the model about the tools it can use and how to call them.<br>
Each tool in our `tool_belt` provides a name and a description of what it does along with the parameters it accepts.<br>
All this information is passed to the model when we blind the tools.<br>

Then the process is as follows :<br>
When the user asks questions, the model :
 - Reads the user's message.
 - Decides, based on the question and the available tool descriptions, whether it can answer directly or if it should call a tool.
 - If it decides a tool is needed, it chooses which tool is most appropiate for the task, based on the tool's name and description.
 - Then the model runs the corresponding tool with the provided arguments and returns the result to the model.
 - The model can then use the tool's output to answer the user's question, possibly calling more tools if needed.

## Task 4: Putting the State in Stateful

Earlier we used this phrasing:

`coordinated multi-actor and stateful applications`

So what does that "stateful" mean?

To put it simply - we want to have some kind of object which we can pass around our application that holds information about what the current situation (state) is. Since our system will be constructed of many parts moving in a coordinated fashion - we want to be able to ensure we have some commonly understood idea of that state.

LangGraph leverages a `StatefulGraph` which uses an `AgentState` object to pass information between the various nodes of the graph.

There are more options than what we'll see below - but this `AgentState` object is one that is stored in a `TypedDict` with the key `messages` and the value is a `Sequence` of `BaseMessages` that will be appended to whenever the state changes.

Let's think about a simple example to help understand exactly what this means (we'll simplify a great deal to try and clearly communicate what state is doing):

1. We initialize our state object:
  - `{"messages" : []}`
2. Our user submits a query to our application.
  - New State: `HumanMessage(#1)`
  - `{"messages" : [HumanMessage(#1)}`
3. We pass our state object to an Agent node which is able to read the current state. It will use the last `HumanMessage` as input. It gets some kind of output which it will add to the state.
  - New State: `AgentMessage(#1, additional_kwargs {"function_call" : "WebSearchTool"})`
  - `{"messages" : [HumanMessage(#1), AgentMessage(#1, ...)]}`
4. We pass our state object to a "conditional node" (more on this later) which reads the last state to determine if we need to use a tool - which it can determine properly because of our provided object!

In [45]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

## Task 5: It's Graphing Time!

Now that we have state, and we have tools, and we have an LLM - we can finally start making our graph!

Let's take a second to refresh ourselves about what a graph is in this context.

Graphs, also called networks in some circles, are a collection of connected objects.

The objects in question are typically called nodes, or vertices, and the connections are called edges.

Let's look at a simple graph.

![image](https://i.imgur.com/2NFLnIc.png)

Here, we're using the coloured circles to represent the nodes and the yellow lines to represent the edges. In this case, we're looking at a fully connected graph - where each node is connected by an edge to each other node.

If we were to think about nodes in the context of LangGraph - we would think of a function, or an LCEL runnable.

If we were to think about edges in the context of LangGraph - we might think of them as "paths to take" or "where to pass our state object next".

Let's create some nodes and expand on our diagram.

> NOTE: Due to the tight integration with LCEL - we can comfortably create our nodes in an async fashion!

In [46]:
from langgraph.prebuilt import ToolNode

def call_model(state):
  messages = state["messages"]
  response = model.invoke(messages)
  return {"messages" : [response]}

tool_node = ToolNode(tool_belt)

Now we have two total nodes. We have:

- `call_model` is a node that will...well...call the model
- `tool_node` is a node which can call a tool

Let's start adding nodes! We'll update our diagram along the way to keep track of what this looks like!


In [47]:
from langgraph.graph import StateGraph, END

uncompiled_graph = StateGraph(AgentState)

uncompiled_graph.add_node("agent", call_model)
uncompiled_graph.add_node("action", tool_node)

Let's look at what we have so far:

![image](https://i.imgur.com/md7inqG.png)

Next, we'll add our entrypoint. All our entrypoint does is indicate which node is called first.

In [48]:
uncompiled_graph.set_entry_point("agent")

![image](https://i.imgur.com/wNixpJe.png)

Now we want to build a "conditional edge" which will use the output state of a node to determine which path to follow.

We can help conceptualize this by thinking of our conditional edge as a conditional in a flowchart!

Notice how our function simply checks if there is a "function_call" kwarg present.

Then we create an edge where the origin node is our agent node and our destination node is *either* the action node or the END (finish the graph).

It's important to highlight that the dictionary passed in as the third parameter (the mapping) should be created with the possible outputs of our conditional function in mind. In this case `should_continue` outputs either `"end"` or `"continue"` which are subsequently mapped to the action node or the END node.

In [49]:
def should_continue(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  return END

uncompiled_graph.add_conditional_edges(
    "agent",
    should_continue
)

Let's visualize what this looks like.

![image](https://i.imgur.com/8ZNwKI5.png)

Finally, we can add our last edge which will connect our action node to our agent node. This is because we *always* want our action node (which is used to call our tools) to return its output to our agent!

In [50]:
uncompiled_graph.add_edge("action", "agent")

Let's look at the final visualization.

![image](https://i.imgur.com/NWO7usO.png)

All that's left to do now is to compile our workflow - and we're off!

In [51]:
simple_agent_graph = uncompiled_graph.compile()

#### ❓ Question #2:

Is there any specific limit to how many times we can cycle?

If not, how could we impose a limit to the number of cycles?

##### ✅ Answer:

By default there is not specific limit to the number of cycles / loops the agent can go through in our graph.<br>
With the current configuration, the agent will keep cycling between nodes as long as the logic in our conditional edges keeps returning a node name, and when it will return to the END node, it will then finish.<br>

If we want to add / impose a specific number of cycles as a limit, we could do it by using a counter variable, tracking the number of cycles/steps in our state and in the conditional function called `should_continue` check if the count exceeds our desired maximum, and if it exceeds returns END.<br>

NOTE : We could also use the method this notebook already implement in the Activity 5, mesuring the messages length : *if len(state["messages"]) > 10:*

Following with the counter approach, we could do it using the following code :

> 1 - Frist update the AgentState by adding the `step_count`:

```python
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    step_count: int
```

> 2 - Update the call_mode returning the step_count incremented by 1
```python
def call_model_with_counter(state):
    messages = state["messages"]
    response = model.invoke(messages)
    return {
        "messages": [response],
        "step_count": state["step_count"] + 1
    }

uncompiled_graph.add_node("agent", call_model_with_counter)
```

> 3 - Update the tool_node, returning the step_count incremented by 1 
```python
def tool_node_with_counter(state):
    tool_node = ToolNode(tool_belt)
    result = tool_node(state)

    return {
        "messages": result["messages"],
        "step_count": state["step_count"] + 1
    }

uncompiled_graph.add_node("action", tool_node_with_counter)
```

> 4 - Then for the condition, check if the limit is greater than 5, if not continue cycling by calling the action. In case it is greater call the END node.
It will also call the END node in case there is not tool_calls.<br>

```python
def should_continue(state):
    last_message = state["messages"][-1]
    step_count = state.get("step_count", 0)

    if step_count >= 5:
        return END  # Stop if limit reached

    if last_message.tool_calls:
        return "action"

    return END
```



## Using Our Graph

Now that we've created and compiled our graph - we can call it *just as we'd call any other* `Runnable`!

Let's try out a few examples to see how it fairs:

In [52]:
from langchain_core.messages import HumanMessage

inputs = {"messages" : [HumanMessage(content="Who is the current captain of the Winnipeg Jets?")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    print(f'chunk : {chunk}')
    for node, values in chunk.items():
        print(f'node : {node} and values : {values}')
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

chunk : {'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_qUQUTIhHTQ4Ny8MCY1n1CbAU', 'function': {'arguments': '{"query":"current captain of the Winnipeg Jets"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 162, 'total_tokens': 185, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': None, 'id': 'chatcmpl-Bs9ZJEdwDYVU8O5BCNV9QJ6WYryey', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--dfcbabfe-8821-4591-a72a-097054f1b8cf-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current captain of the Winnipeg Jets'}, 'id': 'call_qUQUTIhHTQ4Ny8MCY1n1CbAU', 'type': 'tool

Let's look at what happened:

1. Our state object was populated with our request
2. The state object was passed into our entry point (agent node) and the agent node added an `AIMessage` to the state object and passed it along the conditional edge
3. The conditional edge received the state object, found the "tool_calls" `additional_kwarg`, and sent the state object to the action node
4. The action node added the response from the OpenAI function calling endpoint to the state object and passed it along the edge to the agent node
5. The agent node added a response to the state object and passed it along the conditional edge
6. The conditional edge received the state object, could not find the "tool_calls" `additional_kwarg` and passed the state object to END where we see it output in the cell above!

Now let's look at an example that shows a multiple tool usage - all with the same flow!

In [53]:
inputs = {"messages" : [HumanMessage(content="Search Arxiv for the QLoRA paper, then search each of the authors to find out their latest Tweet using Tavily!")]}

##### ✅ Oringal Code
#async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
#    for node, values in chunk.items():
#        print(f"Receiving update from node: '{node}'")
#        if node == "action":
#          print(f"Tool Used: {values['messages'][0].name}")
#        print(values["messages"])

#        print("\n\n")


##### 🏗️ XTALLET - ADDING SOME MORE PRINTS TO CHECK IF THE TOOLS ARE RUNNING IN PARALLEL
async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        if node == "action":
            # Show all tools in the message
            for i, message in enumerate(values["messages"]):
                if hasattr(message, 'name'):
                    print(f"Tool {i+1}: {message.name}")
                    if message.name == "tavily_search_results_json":
                        print("🔍 Tavily was used!")
                    elif message.name == "arxiv":
                        print("�� Arxiv was used!")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Co36hU8Kkgm9sXKaTFqNQlSp', 'function': {'arguments': '{"query": "QLoRA"}', 'name': 'arxiv'}, 'type': 'function'}, {'id': 'call_KPGdabn0kd1i1nRfJPUyK27e', 'function': {'arguments': '{"query": "latest Tweet of author"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 178, 'total_tokens': 232, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': None, 'id': 'chatcmpl-Bs9ZbSr1J0Mdouy0DQjToZpkwEhHo', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--bc1785a5-b8f6-4996-9b05-db505d9e8a88-0', tool_calls=[{'name': 'arxiv', 'args': {'que

#### 🏗️ Activity #2:

Please write out the steps the agent took to arrive at the correct answer.

##### ✅ Answer:
1. The state object was populated with the user's request.
2. The state object was passed to the entry point (agent node)
3. The agent node analyzed the request and generated multiple `tool_calls` in a single response.<br>
 3.1 - One call to Arxiv to search for the QLoRA paper<br>
 3.2 - Three calls to Tavily to search for tweets from each author, first, second and third.<br>
4. The agent node added an AIMessage with tool_calls to the state object.
5. The conditional edge received the state object.
6. It found the `tool_calls` kwargs in the last message.
7. Since tool calls were present, it sent the state object to the action node.
8. The action node received the state with multiple tool calls.
9. It executed all four tools sequentially.<br>
 9.1 Tool 1 : Arxiv - Found the QLoRA paper with the three authors.<br>
 9.2 Tool 2, 3 and 4 : Tavily - Searched for "latest Tweet of the coresponding author of QLoRA".<br>
10. The agent node received the updated state with all tool results.
11. It processed the information from both Arxiv and Tavily searches.
12. It generated a final response summarizing the paper and confirming it searched for tweets from each author.
13. Finally the conditional edge received the final state object.
14. If could not find any `tool_calls` from kwargs in the last message.
15. Since no more tool calls were needed, it passed the state object to END.
16. The final response was the output showing the agent successfully found the QLoRA paper and searched for tweets from all authors.

The screenshot below also shows the trace and the steps followed using both tools : Arxiv and Tavily.<br>
<img src="screenshots/activity2.png" width="1500" />

# 🤝 Breakout Room #2

## Part 1: LangSmith Evaluator

### Pre-processing for LangSmith

To do a little bit more preprocessing, let's wrap our LangGraph agent in a simple chain.

In [54]:
def convert_inputs(input_object):
  return {"messages" : [HumanMessage(content=input_object["question"])]}

def parse_output(input_state):
  return input_state["messages"][-1].content

agent_chain_with_formatting = convert_inputs | simple_agent_graph | parse_output

In [56]:
agent_chain_with_formatting.invoke({"question" : "What is RAG?"})

"RAG can refer to different concepts depending on the context. Could you please specify whether you're asking about RAG in the context of project management, machine learning, or another field?"

### Task 1: Creating An Evaluation Dataset

Just as we saw last week, we'll want to create a dataset to test our Agent's ability to answer questions.

In order to do this - we'll want to provide some questions and some answers. Let's look at how we can create such a dataset below.

```python
questions = [
    "What optimizer is used in QLoRA?",
    "What data type was created in the QLoRA paper?",
    "What is a Retrieval Augmented Generation system?",
    "Who authored the QLoRA paper?",
    "What is the most popular deep learning framework?",
    "What significant improvements does the LoRA system make?"
]

answers = [
    {"must_mention" : ["paged", "optimizer"]},
    {"must_mention" : ["NF4", "NormalFloat"]},
    {"must_mention" : ["ground", "context"]},
    {"must_mention" : ["Tim", "Dettmers"]},
    {"must_mention" : ["PyTorch", "TensorFlow"]},
    {"must_mention" : ["reduce", "parameters"]},
]
```

#### 🏗️ Activity #3:

Please create a dataset in the above format with at least 5 questions.

##### ✅ Answer:
I have written the 5 new questions directly in the code cell below.

In [57]:
questions = [
    "What is the main advantage of QLoRA over traditional fine-tuning?",
    "What is the name of the model family introduced in QLoRA?",
    "What is the purpose of double quantization in QLoRA?",
    "What benchmark did the QLoRA models outperform?",
    "What is the memory reduction achieved by QLoRA quantization?"
]

answers = [
    {"must_mention" : ["memory", "usage"]},
    {"must_mention" : ["Guanaco"]},
    {"must_mention" : ["quantization", "constants"]},
    {"must_mention" : ["Vicuna", "benchmark"]},
    {"must_mention" : ["4-bit", "quantization"]},
]

Now we can add our dataset to our LangSmith project using the following code which we saw last Thursday!

In [58]:
from langsmith import Client

client = Client()

dataset_name = f"Retrieval Augmented Generation - Evaluation Dataset - {uuid4().hex[0:8]}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Questions about the QLoRA Paper to Evaluate RAG over the same paper."
)

client.create_examples(
    inputs=[{"question" : q} for q in questions],
    outputs=answers,
    dataset_id=dataset.id,
)

{'example_ids': ['32cc5eef-5de2-486d-b14e-d57e621c9680',
  '80dcfb18-aac2-43ca-91fe-b9004a042305',
  '67d794c8-c17b-4a97-aa75-8aeb71bd9f13',
  'ccf64327-8588-4a34-b89e-7e2e77bf2440',
  '0560bb73-28e8-47c9-acb9-287b45109575'],
 'count': 5}

#### ❓ Question #3:

How are the correct answers associated with the questions?

> NOTE: Feel free to indicate if this is problematic or 

##### ✅ Answer:

The correct answers are associated with the questions by their position in the respective lists: the first question is paired with the first answer, the second with the second, and so on.<br> This positional association means that if the order of either list changes, the mapping between questions and answers could become incorrect.

> Is this problematic ?

Yes, this approach can be problematic because it is easy to accidentally misalign the questions and answers if the lists are edited separately. A more robust approach would be to use a data structure that explicitly links each question to its answer.

> proposed solution, creating the dataset associating each question with its answer

```python
examples = [
    {
        "question": "What is the main advantage of QLoRA over traditional fine-tuning?",
        "answer": {"must_mention": ["memory", "usage"]}
    },
    {
        "question": "What is the name of the model family introduced in QLoRA?",
        "answer": {"must_mention": ["Guanaco"]}
    },
    {
        "question": "What is the purpose of double quantization in QLoRA?",
        "answer": {"must_mention": ["quantization", "constants"]}
    },
    {
        "question": "What benchmark did the QLoRA models outperform?",
        "answer": {"must_mention": ["Vicuna", "benchmark"]}
    },
    {
        "question": "What is the memory reduction achieved by QLoRA quantization?",
        "answer": {"must_mention": ["4-bit", "quantization"]}
    },
]
```
> Then create the dataset as follows :
```python
inputs = [{"question": ex["question"]} for ex in examples]
outputs = [ex["answer"] for ex in examples]

client = Client()

dataset_name = f"Retrieval Augmented Generation - Evaluation Dataset - {uuid4().hex[0:8]}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Questions about the QLoRA Paper to Evaluate RAG over the same paper."
)

client.create_examples(
    inputs=inputs,
    outputs=outputs,
    dataset_id=dataset.id,
)
```

### Task 2: Adding Evaluators

Now we can add a custom evaluator to see if our responses contain the expected information.

We'll be using a fairly naive exact-match process to determine if our response contains specific strings.

In [59]:
from langsmith.evaluation import EvaluationResult, run_evaluator

@run_evaluator
def must_mention(run, example) -> EvaluationResult:
    prediction = run.outputs.get("output") or ""
    required = example.outputs.get("must_mention") or []
    score = all(phrase in prediction for phrase in required)
    return EvaluationResult(key="must_mention", score=score)

#### ❓ Question #4:

What are some ways you could improve this metric as-is?

> NOTE: Alternatively you can suggest where gaps exist in this method.

##### ✅ Answer:

This is the definition for the metric `must_mention`:<br>
The `must_mention` metric is an automatic evaluation method designed to check whether a model’s output contains specific required phrases.

That said, some ways to improve this metric include making the phrase matching case-insensitive, allowing for partial or fuzzy matches, and using semantic similarity rather than exact string matching.<br><br> 
The current method only checks for exact phrase presence, which can miss correct answers that use synonyms or paraphrasing. Additionally, it does not consider context, spelling variations, or whether the required phrases are used appropriately.<br> 
Reporting which required phrases are missing can also help with error analysis.

Task 3: Evaluating

All that is left to do is evaluate our agent's response!

In [60]:
experiment_results = client.evaluate(
    agent_chain_with_formatting,
    data=dataset_name,
    evaluators=[must_mention],
    experiment_prefix=f"Search Pipeline - Evaluation - {uuid4().hex[0:4]}",
    metadata={"version": "1.0.0"},
)

View the evaluation results for experiment: 'Search Pipeline - Evaluation - 4048-98da44b0' at:
https://smith.langchain.com/o/c21c7da9-346c-4a28-b19c-bb6d2faffe60/datasets/c2a82cec-ee1c-4d47-9625-710b1297c370/compare?selectedSessions=974c9595-28a4-4cfc-b0bf-6e8a78be52e5




0it [00:00, ?it/s]

In [61]:
experiment_results

<ExperimentResults Search Pipeline - Evaluation - 4048-98da44b0>

## Part 2: LangGraph with Helpfulness:

### Task 3: Adding Helpfulness Check and "Loop" Limits

Now that we've done evaluation - let's see if we can add an extra step where we review the content we've generated to confirm if it fully answers the user's query!

We're going to make a few key adjustments to account for this:

1. We're going to add an artificial limit on how many "loops" the agent can go through - this will help us to avoid the potential situation where we never exit the loop.
2. We'll add to our existing conditional edge to obtain the behaviour we desire.

First, let's define our state again - we can check the length of the state object, so we don't need additional state for this.

In [62]:
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

Now we can set our graph up! This process will be almost entirely the same - with the inclusion of one additional node/conditional edge!

#### 🏗️ Activity #5:

Please write markdown for the following cells to explain what each is doing.

##### ✅ Answer:
This code creates a state graph for an agent, adding two nodes, one for calling the model (the `agent`) and another for performing an action with a tool (the `action`)

In [65]:
graph_with_helpfulness_check = StateGraph(AgentState)

graph_with_helpfulness_check.add_node("agent", call_model)
graph_with_helpfulness_check.add_node("action", tool_node)

##### ✅ Answer:

This code sets the agent node as the starting point of the state graph.

In [66]:
graph_with_helpfulness_check.set_entry_point("agent")

##### ✅ Answer:

This function called `tool_call_or_helpful` decides the next step in the agent's workflow :<br>
If the last message includes tool calls, it returns `actions`.<br>

If there are more than 10 messages, it returns `END`.<br>

Otherwhise it uses a language model to check if the final response is helpful.<br>
If the final response is helpful, it returns `end`.<br>
If the final response is not helpful, it returns `continue`.<br><br>

In summary, this function controls the agent’s flow by checking for tool usage, limiting the number of steps, and using an LLM to assess the helpfulness of the agent’s response.

In [67]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def tool_call_or_helpful(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  initial_query = state["messages"][0]
  final_response = state["messages"][-1]

  if len(state["messages"]) > 10:
    return "END"

  prompt_template = """\
  Given an initial query and a final response, determine if the final response is extremely helpful or not. Please indicate helpfulness with a 'Y' and unhelpfulness as an 'N'.

  Initial Query:
  {initial_query}

  Final Response:
  {final_response}"""

  helpfullness_prompt_template = PromptTemplate.from_template(prompt_template)

  helpfulness_check_model = ChatOpenAI(model="gpt-4.1-mini")

  helpfulness_chain = helpfullness_prompt_template | helpfulness_check_model | StrOutputParser()

  helpfulness_response = helpfulness_chain.invoke({"initial_query" : initial_query.content, "final_response" : final_response.content})

  if "Y" in helpfulness_response:
    return "end"
  else:
    return "continue"

#### 🏗️ Activity #4:

Please write what is happening in our `tool_call_or_helpful` function!

##### ✅ Answer:

It is adding conditional edges to the Graph, which is used to model the flow of a conversational agent.<br>
The function `tool_call_or_helpful`receives the current state and decides which of the three options (continue, action or end) applies, based on its internal logic.<br>

Depending on what it returns, the graph transitions to a different node:
- If it returns "continue", the flow goes back to the "agent" node.
- If it returns "action", the flow goes to the "action" node.
- If it returns "end", the flow ends (goes to END).

In summary the function `tool_call_or_helpful` analyzes the current state and decides whether the agent should keep thinking, execute an action, or end the conversation.

In [68]:
graph_with_helpfulness_check.add_conditional_edges(
    "agent",
    tool_call_or_helpful,
    {
        "continue" : "agent",
        "action" : "action",
        "end" : END
    }
)

##### ✅ Answer:

This line adds an edge from the `action` node back to the `agent`, allowing the agent to continue its workflow after performing an action.

In [69]:
graph_with_helpfulness_check.add_edge("action", "agent")

##### ✅ Answer:

This line compiles the state graph into an executable agent called `agent_with_helpfulness_check`.

In [70]:
agent_with_helpfulness_check = graph_with_helpfulness_check.compile()

##### ✅ Answer:

This code sends and input message to the compiled agent and asynchronously streams updates as the agent processes the input. For each update, it prints which node produced the update and displays the current messages.

In [71]:
inputs = {"messages" : [HumanMessage(content="Related to machine learning, what is LoRA? Also, who is Tim Dettmers? Also, what is Attention?")]}

async for chunk in agent_with_helpfulness_check.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_NuQU78UEsQ6x5rrsECa3fVoe', 'function': {'arguments': '{"query": "LoRA machine learning"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}, {'id': 'call_rLm57XiJaRC0t14jTMI8ZgxV', 'function': {'arguments': '{"query": "Tim Dettmers"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}, {'id': 'call_x43QyntJIOpooXVmVBk3eEYD', 'function': {'arguments': '{"query": "Attention in machine learning"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 177, 'total_tokens': 256, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': None, 'id': 'chatcmpl-Bs9

### Task 4: LangGraph for the "Patterns" of GenAI

Let's ask our system about the 4 patterns of Generative AI:

1. Prompt Engineering
2. RAG
3. Fine-tuning
4. Agents

In [72]:
patterns = ["prompt engineering", "RAG", "fine-tuning", "LLM-based agents"]

In [73]:
for pattern in patterns:
  what_is_string = f"What is {pattern} and when did it break onto the scene??"
  inputs = {"messages" : [HumanMessage(content=what_is_string)]}
  messages = agent_with_helpfulness_check.invoke(inputs)
  print(messages["messages"][-1].content)
  print("\n\n")

Prompt engineering is the process of designing and refining prompts to effectively communicate with AI language models, such as GPT, to obtain desired responses. It involves crafting prompts that are clear, specific, and contextually appropriate to guide the AI's output in a useful and accurate manner.

Prompt engineering has gained significant prominence with the rise of large language models (LLMs) like GPT-3, which were released around 2020. As these models became more capable and widely adopted, the importance of effectively interacting with them through well-designed prompts also grew. The term and practice of prompt engineering started to break onto the scene around 2021-2022, coinciding with the increasing popularity of LLMs and their applications across various domains. It has since become a crucial skill for developers, researchers, and users working with AI language models.



RAG, which stands for Retrieval-Augmented Generation, is a technique in natural language processing 